# Aria Code Patterns & Snippets Library

**Copy-paste ready code examples for common patterns in Aria.**

This notebook contains working code snippets for frequent tasks. All examples follow Aria conventions and best practices.


## Pattern 1: Detecting & Using Chat Providers

**Problem:** How do I use chat providers in my code?
**Location:** `shared/chat_providers.py`
**Complexity:** Medium

### Basic Provider Detection

```python
from shared.chat_providers import detect_provider, PROVIDER_CHAIN

# Auto-detect provider (tries LMStudio → Ollama → Azure → OpenAI → Local)
provider = detect_provider()

# Explicitly choose provider
provider = detect_provider(force_provider="azure")

# Get provider chain (ordered list of available providers)
chain = PROVIDER_CHAIN()  # Returns: ["lmstudio", "ollama", "azure", ...]
```

### Using Provider for Chat

```python
import asyncio
from shared.chat_providers import detect_provider

async def chat_with_user(prompt: str):
    provider = detect_provider()

    # Get single completion
    response = await provider.complete(prompt)
    print(f"Response: {response}")

    # Or stream tokens one by one
    async for token in provider.stream(prompt):
        print(token, end="", flush=True)
    print()  # Newline at end

# Run async function
asyncio.run(chat_with_user("Hello, how are you?"))
```

### Provider with Error Handling

```python
from shared.chat_providers import detect_provider, ChatProvider

async def safe_chat(prompt: str, max_retries: int = 3):
    try:
        provider = detect_provider()
        response = await provider.complete(prompt)
        return response
    except Exception as e:
        print(f"Provider error: {e}")
        # Fallback to local provider
        from shared.chat_providers import LocalProvider
        fallback = LocalProvider()
        response = await fallback.complete(prompt)
        return response
```

### Checking Provider Configuration

```python
import os
from shared.config import Settings

settings = Settings()

# Check current provider
print(f"Active provider: {settings.provider}")

# Check available env vars
print(f"Azure available: {'AZURE_OPENAI_API_KEY' in os.environ}")
print(f"OpenAI available: {'OPENAI_API_KEY' in os.environ}")

# Get provider priority
print(f"Provider chain: {settings.provider_chain()}")
```


## Pattern 2: Working with Semantic Memory

**Problem:** How do I add semantic memory to my chat flow?
**Location:** `shared/chat_memory.py`
**Complexity:** Medium

### Generate Embeddings

```python
from shared.chat_memory import generate_embedding, fetch_similar, store_memory

async def add_context_to_prompt(user_message: str, session_id: str):
    # 1. Generate embedding for user message
    embedding = await generate_embedding(user_message)

    # 2. Fetch similar messages from memory (top 5)
    similar_messages = await fetch_similar(
        query=user_message,
        session_id=session_id,
        k=5
    )

    # 3. Build context string
    context = "\n".join([
        f"Similar past message: {msg['content']}"
        for msg in similar_messages
    ])

    # 4. Augment original prompt
    augmented_prompt = f"""
Context from memory:
{context}

User query:
{user_message}
"""

    return augmented_prompt
```

### Store Message in Memory

```python
from shared.chat_memory import store_memory

async def chat_and_remember(
    session_id: str,
    user_message: str,
    assistant_response: str
):
    # Store user message
    await store_memory(
        session_id=session_id,
        role="user",
        content=user_message,
        metadata={"timestamp": "2024-01-01T12:00:00Z"}
    )

    # Store assistant response
    await store_memory(
        session_id=session_id,
        role="assistant",
        content=assistant_response,
        metadata={"model": "gpt-4"}
    )
```

### Memory with Database Fallback

```python
from shared.chat_memory import fetch_similar, store_memory
import logging

async def resilient_memory(query: str, session_id: str):
    try:
        # Try primary memory store
        results = await fetch_similar(query, session_id, k=5)
        return results
    except Exception as e:
        logging.warning(f"Memory fetch failed: {e}, using empty context")
        # Return empty list so chat continues without memory
        return []
```


## Pattern 3: Building FastAPI Endpoints

**Problem:** How do I add a new API endpoint to function_app.py?
**Location:** `function_app.py`
**Complexity:** Low-Medium

### Simple GET Endpoint

```python
from azure.functions import HttpRequest, HttpResponse
import azure.functions as func

# Create a function app
app = func.FunctionApp()

@app.function_name("MyGetEndpoint")
@app.route("api/my-endpoint", methods=["GET"])
async def get_endpoint(req: HttpRequest) -> HttpResponse:
    try:
        # Get query parameters
        param = req.params.get("key")

        # Do something
        result = {"status": "ok", "value": param}

        return HttpResponse(
            json.dumps(result),
            status_code=200,
            mimetype="application/json"
        )
    except Exception as e:
        return HttpResponse(
            json.dumps({"error": str(e)}),
            status_code=500,
            mimetype="application/json"
        )
```

### POST Endpoint with JSON Body

```python
@app.function_name("MyPostEndpoint")
@app.route("api/my-endpoint", methods=["POST"])
async def post_endpoint(req: HttpRequest) -> HttpResponse:
    try:
        # Parse JSON body
        data = req.get_json()
        message = data.get("message")

        # Validate input
        if not message:
            return HttpResponse(
                json.dumps({"error": "message required"}),
                status_code=400,
                mimetype="application/json"
            )

        # Process
        result = await process_message(message)

        return HttpResponse(
            json.dumps(result),
            status_code=200,
            mimetype="application/json"
        )
    except ValueError as e:
        return HttpResponse(
            json.dumps({"error": str(e)}),
            status_code=400,
            mimetype="application/json"
        )
    except Exception as e:
        return HttpResponse(
            json.dumps({"error": str(e)}),
            status_code=500,
            mimetype="application/json"
        )
```

### Streaming SSE Response

```python
@app.function_name("MyStreamEndpoint")
@app.route("api/my-stream", methods=["POST"])
async def stream_endpoint(req: HttpRequest) -> HttpResponse:
    async def generate_stream():
        try:
            data = req.get_json()
            prompt = data.get("prompt")

            # Stream tokens from provider
            async for token in some_async_generator(prompt):
                # Format as SSE
                yield f"data: {json.dumps({'token': token})}\n\n"

            # Send done marker
            yield "data: [DONE]\n\n"
        except Exception as e:
            yield f"data: {json.dumps({'error': str(e)})}\n\n"

    return HttpResponse(
        generate_stream(),
        status_code=200,
        mimetype="text/event-stream"
    )
```


## Pattern 4: Quantum Circuit Development

**Problem:** How do I create and run a quantum circuit?
**Location:** `ai-projects/quantum-ml/src/quantum_llm/`
**Complexity:** High

### Simple Quantum Circuit

```python
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

# Create quantum circuit
qr = QuantumRegister(3, "q")
cr = ClassicalRegister(3, "c")
circuit = QuantumCircuit(qr, cr)

# Add quantum gates
circuit.h(qr[0])  # Hadamard on qubit 0
circuit.cx(qr[0], qr[1])  # CNOT on qubits 0,1
circuit.measure(qr, cr)  # Measure all qubits

# Run on simulator
simulator = AerSimulator()
job = simulator.run(circuit, shots=1000)
result = job.result()

# Get results
counts = result.get_counts(circuit)
print(f"Measurement results: {counts}")
```

### Quantum Circuit with Safety Gates

```python
from ai_projects.quantum_ml.src.quantum_llm.config import (
    SIMULATOR_ONLY,
    AZURE_CONFIRM_COST,
    MAX_SHOTS,
    CIRCUIT_DEPTH_LIMIT
)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

async def safe_quantum_run(circuit: QuantumCircuit, shots: int = 100):
    # Safety checks
    if shots > MAX_SHOTS:
        raise ValueError(f"Shots ({shots}) exceeds limit ({MAX_SHOTS})")

    if circuit.depth() > CIRCUIT_DEPTH_LIMIT:
        raise ValueError(f"Circuit too deep ({circuit.depth()} > {CIRCUIT_DEPTH_LIMIT})")

    # Check provider
    if SIMULATOR_ONLY:
        # Only run on simulator
        provider = "simulator"
    else:
        provider = "auto"  # Can use real QPU

    # Run circuit
    simulator = AerSimulator()
    job = simulator.run(circuit, shots=shots)
    result = job.result()

    return result.get_counts(circuit)
```

### Quantum Embedding Circuit

```python
from qiskit import QuantumCircuit, QuantumRegister
import numpy as np

def create_embedding_circuit(data: list, num_qubits: int = 4):
    """Create circuit for embedding classical data into quantum state"""
    qr = QuantumRegister(num_qubits, "q")
    circuit = QuantumCircuit(qr)

    # Normalize data
    data = np.array(data)
    data = data / np.linalg.norm(data)

    # Encode data using angle encoding
    for i, angle in enumerate(data[:num_qubits]):
        circuit.ry(angle * np.pi, qr[i])

    # Entangle qubits
    for i in range(num_qubits - 1):
        circuit.cx(qr[i], qr[i + 1])

    return circuit
```


## Pattern 5: LoRA Fine-Tuning

**Problem:** How do I fine-tune a model with LoRA?
**Location:** `ai-projects/lora-training/`
**Complexity:** High

### Basic LoRA Setup

```python
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load base model
model_name = "microsoft/phi-3.5-mini"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Alpha
    target_modules=["q_proj", "v_proj"],  # Which layers to apply LoRA to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())  # See how many params are trainable
```

### LoRA Training Loop

```python
from transformers import Trainer, TrainingArguments
import torch

# Prepare training arguments
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    logging_steps=10,
    save_steps=100,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# Train
trainer.train()

# Save adapter
model.save_pretrained("./my_adapter")
```

### Load and Use LoRA Model

```python
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# Load model with LoRA adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    "microsoft/phi-3.5-mini",
    adapter_path="./my_adapter"
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3.5-mini")

# Generate text
inputs = tokenizer("Hello, how are you?", return_tensors="pt")
outputs = model.generate(**inputs, max_length=100)
print(tokenizer.decode(outputs[0]))
```


## Pattern 6: Database Operations

**Problem:** How do I read/write to the database?
**Location:** `shared/sql_engine.py`
**Complexity:** Medium

### Basic Database Connection

```python
from shared.sql_engine import get_session, run_async
import logging

async def save_to_database(user_id: str, message: str):
    try:
        session = get_session()

        # Create record
        record = ChatMessage(
            user_id=user_id,
            content=message,
            timestamp=datetime.now()
        )

        session.add(record)
        session.commit()

        return {"status": "saved", "id": record.id}
    except Exception as e:
        logging.error(f"Database error: {e}")
        session.rollback()
        return {"error": str(e)}
    finally:
        session.close()
```

### Query from Database

```python
from shared.sql_engine import get_session
from sqlalchemy import select

async def get_recent_messages(user_id: str, limit: int = 10):
    session = get_session()
    try:
        # Query
        query = select(ChatMessage).where(
            ChatMessage.user_id == user_id
        ).order_by(
            ChatMessage.timestamp.desc()
        ).limit(limit)

        result = session.execute(query)
        messages = result.scalars().all()

        return [
            {"content": msg.content, "timestamp": msg.timestamp}
            for msg in messages
        ]
    finally:
        session.close()
```

### Database with Connection Pool

```python
from shared.sql_engine import check_pool_status
import os

# Set pool size in environment
os.environ["QAI_SQL_POOL_SIZE"] = "20"

# Check pool status
async def monitor_database():
    status = check_pool_status()
    print(f"Connections available: {status['available']}")
    print(f"Connections used: {status['used']}")
    print(f"Pool saturation: {status['saturation']}%")

    if status['saturation'] > 80:
        logging.warning("Database pool near saturation!")
```


## Pattern 7: Testing Your Code

**Problem:** How do I write tests for my feature?
**Location:** `tests/unit/`, `tests/integration/`
**Complexity:** Medium

### Basic Unit Test

```python
import pytest
from my_module import process_data

@pytest.mark.unit
def test_process_data_valid():
    """Test with valid input"""
    result = process_data("hello world")
    assert result == "HELLO WORLD"

@pytest.mark.unit
def test_process_data_empty():
    """Test with empty input"""
    result = process_data("")
    assert result == ""

@pytest.mark.unit
def test_process_data_special_chars():
    """Test with special characters"""
    result = process_data("@#$")
    assert result == "@#$"
```

### Async Test

```python
import pytest
from my_module import async_process

@pytest.mark.unit
@pytest.mark.asyncio
async def test_async_process():
    """Test async function"""
    result = await async_process("test")
    assert result is not None

@pytest.mark.unit
@pytest.mark.asyncio
async def test_async_process_error_handling():
    """Test error handling in async function"""
    with pytest.raises(ValueError):
        await async_process(None)
```

### Integration Test

```python
import pytest
from shared.sql_engine import get_session
from my_module import save_and_retrieve

@pytest.mark.integration
async def test_save_and_retrieve():
    """Test save and retrieve from database"""
    session = get_session()

    # Save
    result = await save_and_retrieve("test message")

    # Retrieve
    saved = session.query(ChatMessage).filter_by(
        content="test message"
    ).first()

    assert saved is not None
    assert saved.content == "test message"

    # Cleanup
    session.delete(saved)
    session.commit()
```

### Mock Dependencies

```python
import pytest
from unittest.mock import AsyncMock, patch
from my_module import chat_with_provider

@pytest.mark.unit
@pytest.mark.asyncio
async def test_chat_with_mock_provider():
    """Test chat with mocked provider"""
    mock_provider = AsyncMock()
    mock_provider.complete.return_value = "Mocked response"

    with patch("my_module.detect_provider", return_value=mock_provider):
        result = await chat_with_provider("Hello")
        assert result == "Mocked response"
        mock_provider.complete.assert_called_once()
```


## Pattern 8: Error Handling & Logging

**Problem:** How do I handle errors properly?
**Location:** Throughout codebase
**Complexity:** Low

### Graceful Error Handling

```python
import logging
from typing import Optional

logger = logging.getLogger(__name__)

async def fetch_with_fallback(primary_url: str, fallback_url: str):
    """Try primary, fallback to secondary on error"""
    try:
        # Try primary
        result = await fetch(primary_url)
        return result
    except ConnectionError as e:
        logger.warning(f"Primary URL failed: {e}, using fallback")
        try:
            # Try fallback
            result = await fetch(fallback_url)
            return result
        except ConnectionError as e2:
            logger.error(f"Both URLs failed: {e2}")
            return None
```

### Structured Logging

```python
import logging
import json
from datetime import datetime

logger = logging.getLogger(__name__)

def log_event(event_type: str, details: dict):
    """Log structured event"""
    log_data = {
        "timestamp": datetime.now().isoformat(),
        "event_type": event_type,
        "details": details
    }
    logger.info(json.dumps(log_data))

# Usage
log_event("user_chat", {"user_id": "123", "message_length": 50})
```

### Telemetry Integration

```python
from shared.telemetry import log_exception, log_metric

async def safe_operation(user_id: str):
    try:
        result = await some_operation()
        log_metric("operation_success", {"user_id": user_id})
        return result
    except Exception as e:
        log_exception(f"Operation failed for user {user_id}", e)
        raise
```


## 📚 Using These Patterns

### How to Find the Right Pattern

1. Search this notebook with Ctrl+F for your task
2. Copy the code snippet
3. Modify for your specific needs
4. Test with example in pattern

### Best Practices

- Always include error handling (try/except)
- Use logging for debugging
- Test before committing
- Follow the existing code style
- Document non-obvious code

### Common Modifications

- Change function names to match your use case
- Add parameters as needed
- Extend error handling as required
- Add logging for critical operations

### When to Create New Patterns

If you find yourself writing the same code 3+ times:

1. Extract into a reusable function
2. Add to `shared/` module
3. Document in this notebook
4. Link to implementation


## 📚 Using These Patterns

### How to Find the Right Pattern

1. Search this notebook with Ctrl+F for your task
2. Copy the code snippet
3. Modify for your specific needs
4. Test with example in pattern

### Best Practices

- Always include error handling (try/except)
- Use logging for debugging
- Test before committing
- Follow the existing code style
- Document non-obvious code

### Common Modifications

- Change function names to match your use case
- Add parameters as needed
- Extend error handling as required
- Add logging for critical operations

### When to Create New Patterns

If you find yourself writing the same code 3+ times:

1. Extract into a reusable function
2. Add to `shared/` module
3. Document in this notebook
4. Link to implementation
